# 04 - Brutal Roaster 🔥

This notebook implements a **savage, no-holds-barred** CV critique model.

## Characteristics
- Brutally honest and hilarious
- Roasts with humor and wit
- Calls out everything wrong
- Entertainment value meets harsh truth
- Uses high temperature (0.8-0.95) for maximum creativity and humor

⚠️ **Warning**: This model is intentionally harsh for comedic effect!

---

## Setup

In [ ]:
import pandas as pd
import json
from pathlib import Path
import google.generativeai as genai
from datetime import datetime

# Configure Gemini API
GEMINI_API_KEY = "YOUR_API_KEY_HERE"  # Replace with your key
genai.configure(api_key=GEMINI_API_KEY)

## Load Data and Helper Functions

In [ ]:
# Load dataset
df = pd.read_csv('../data/resume_data.csv')

# Load test CV indices
with open('../data/test_cv_indices.json', 'r') as f:
    test_data = json.load(f)
    test_cv_indices = test_data['indices']

print(f"Loaded {len(df)} resumes")
print(f"Test CVs: {test_cv_indices}")

In [ ]:
# CV formatting function
def format_cv_for_llm(resume_row):
    """
    Format a resume row into a readable text for LLM processing.
    """
    cv_text = []
    
    if pd.notna(resume_row.get('career_objective')):
        cv_text.append(f"CAREER OBJECTIVE:\n{resume_row['career_objective']}")
    
    if pd.notna(resume_row.get('skills')):
        cv_text.append(f"\nSKILLS:\n{resume_row['skills']}")
    
    education_parts = []
    if pd.notna(resume_row.get('educational_institution_name')):
        education_parts.append(f"Institution: {resume_row['educational_institution_name']}")
    if pd.notna(resume_row.get('degree_names')):
        education_parts.append(f"Degree: {resume_row['degree_names']}")
    if pd.notna(resume_row.get('major_field_of_studies')):
        education_parts.append(f"Major: {resume_row['major_field_of_studies']}")
    if pd.notna(resume_row.get('passing_years')):
        education_parts.append(f"Year: {resume_row['passing_years']}")
    
    if education_parts:
        cv_text.append(f"\nEDUCATION:\n" + "\n".join(education_parts))
    
    work_parts = []
    if pd.notna(resume_row.get('professional_company_names')):
        work_parts.append(f"Company: {resume_row['professional_company_names']}")
    if pd.notna(resume_row.get('positions')):
        work_parts.append(f"Position: {resume_row['positions']}")
    if pd.notna(resume_row.get('start_dates')):
        work_parts.append(f"Period: {resume_row['start_dates']}")
        if pd.notna(resume_row.get('end_dates')):
            work_parts.append(f" to {resume_row['end_dates']}")
    if pd.notna(resume_row.get('responsibilities')):
        work_parts.append(f"Responsibilities:\n{resume_row['responsibilities']}")
    
    if work_parts:
        cv_text.append(f"\nWORK EXPERIENCE:\n" + "\n".join(work_parts))
    
    if pd.notna(resume_row.get('languages')):
        cv_text.append(f"\nLANGUAGES:\n{resume_row['languages']}")
    
    if pd.notna(resume_row.get('certification_skills')):
        cv_text.append(f"\nCERTIFICATIONS:\n{resume_row['certification_skills']}")
    
    return "\n".join(cv_text)

## Brutal Roaster Prompt Design

In [ ]:
BRUTAL_SYSTEM_PROMPT = """You are a savage CV roaster with a sharp wit and no filter.

Your approach:
1. Be brutally honest and hilariously savage
2. Use humor, sarcasm, and wit to roast every weakness
3. Call out buzzwords, clichés, and BS mercilessly
4. Make it funny but still insightful
5. Don't hold back - this is entertainment

Keep your roast:
- Savage but clever
- Funny and entertaining
- Honest to the point of painful
- Creative with your insults
- Still somewhat constructive (through the burns)

Structure your response:
🔥 OPENING ROAST: Hit them hard right away
💀 CAREER OBJECTIVE AUTOPSY: Tear apart their career goals
🤡 SKILLS SECTION COMEDY: Mock their skill claims
📉 EXPERIENCE REALITY CHECK: Expose the truth
⚰️ FATAL FLAWS: The worst offenses
🎤 MIC DROP: Final devastating verdict

Use humor devices like:
- Sarcasm and irony
- Exaggeration for effect
- Pop culture references
- Metaphors and comparisons
- Dark humor (but keep it about the CV, not personal attacks)
"""

def create_brutal_prompt(cv_text):
    """Create a brutal roasting prompt."""
    return f"""Roast this CV with maximum savagery. Be hilarious, brutal, and creative:

{cv_text}

Unleash your most savage critique following the structure in the system prompt. Make it hurt (in a funny way)."""

## Temperature Tuning Experiments

Let's test high temperatures for maximum creativity and humor.

In [ ]:
def roast_cv(cv_text, temperature=0.9, model_name="gemini-1.5-flash"):
    """
    Generate CV critique using Gemini.
    
    Args:
        cv_text: Formatted CV text
        temperature: Controls randomness (0.0-1.0)
        model_name: Gemini model to use
    
    Returns:
        str: Generated critique
    """
    model = genai.GenerativeModel(
        model_name=model_name,
        generation_config=genai.GenerationConfig(
            temperature=temperature,
            top_p=0.95,
            top_k=40,
            max_output_tokens=1536,  # More tokens for creative roasts
        )
    )
    
    full_prompt = f"{BRUTAL_SYSTEM_PROMPT}\n\n{create_brutal_prompt(cv_text)}"
    
    response = model.generate_content(full_prompt)
    return response.text

# Test with first CV
test_cv = format_cv_for_llm(df.iloc[test_cv_indices[0]])
print("Test CV:")
print("="*80)
print(test_cv[:500] + "...")
print("="*80)

### Experiment 1: High Temperature (0.8)
Creative but somewhat controlled

In [ ]:
print("🌡️ Temperature: 0.8 (Creative)")
print("="*80)
result_temp_08 = roast_cv(test_cv, temperature=0.8)
print(result_temp_08)
print("\n" + "="*80)

### Experiment 2: Very High Temperature (0.9)
Maximum creativity and humor

In [ ]:
print("🌡️ Temperature: 0.9 (Maximum Creativity)")
print("="*80)
result_temp_09 = roast_cv(test_cv, temperature=0.9)
print(result_temp_09)
print("\n" + "="*80)

### Experiment 3: Extreme Temperature (0.95)
Wild and unpredictable

In [ ]:
print("🌡️ Temperature: 0.95 (Extreme)")
print("="*80)
result_temp_095 = roast_cv(test_cv, temperature=0.95)
print(result_temp_095)
print("\n" + "="*80)

## Select Optimal Temperature

**Recommended: 0.9 for brutal roasting** - provides:
- Maximum creativity and humor
- Varied, entertaining roasts
- Still coherent and focused
- Good balance of savage and funny

In [ ]:
# Set optimal temperature
OPTIMAL_TEMPERATURE = 0.9

print(f"✅ Selected optimal temperature: {OPTIMAL_TEMPERATURE}")

## Generate Results for Test CVs

In [ ]:
# Create results directory
results_dir = Path('../results/brutal_roaster')
results_dir.mkdir(parents=True, exist_ok=True)

# Process each test CV
results = []

for idx in test_cv_indices:
    print(f"\n{'='*80}")
    print(f"Processing CV #{idx}")
    print(f"{'='*80}\n")
    
    # Format CV
    cv_text = format_cv_for_llm(df.iloc[idx])
    
    # Generate critique
    critique = roast_cv(cv_text, temperature=OPTIMAL_TEMPERATURE)
    
    # Save result
    result = {
        'cv_index': idx,
        'model': 'brutal_roaster',
        'temperature': OPTIMAL_TEMPERATURE,
        'timestamp': datetime.now().isoformat(),
        'cv_text': cv_text,
        'critique': critique
    }
    
    results.append(result)
    
    # Save to file
    output_file = results_dir / f"cv_{idx}_brutal.json"
    with open(output_file, 'w') as f:
        json.dump(result, f, indent=2)
    
    # Display result
    print(f"📝 Original CV:")
    print("-"*80)
    print(cv_text)
    print(f"\n🔥 Brutal Roast:")
    print("-"*80)
    print(critique)
    print(f"\n✅ Saved to: {output_file}")
    print(f"\n{'='*80}\n")

print(f"\n✅ Generated {len(results)} brutal roasts")
print("\n🔥 WARNING: These roasts are intentionally harsh for comedic effect! 🔥")

## Roast Quality Check

Let's analyze the characteristics of our brutal roasts.

In [ ]:
print("📊 ROAST STATISTICS")
print("="*80)

for result in results:
    cv_idx = result['cv_index']
    critique = result['critique']
    
    print(f"\nCV #{cv_idx}:")
    print(f"  Length: {len(critique)} characters")
    print(f"  Words: {len(critique.split())} words")
    print(f"  Lines: {len(critique.split(chr(10)))} lines")
    
    # Check for humor indicators
    humor_indicators = ['😂', '🤣', '💀', '🔥', '🤡', '⚰️']
    emoji_count = sum(critique.count(emoji) for emoji in humor_indicators)
    print(f"  Humor emojis: {emoji_count}")

print("\n" + "="*80)

## Summary

This notebook demonstrated:
1. ✅ Savage, humorous CV critique prompt design
2. ✅ High temperature tuning experiments (0.8, 0.9, 0.95)
3. ✅ Selection of optimal temperature for maximum creativity
4. ✅ Generation of brutal roasts for test CVs
5. ✅ Saving results for comparison

**Key Findings:**
- Higher temperatures (0.9+) produce more creative and varied roasts
- Humor and wit emerge better with increased randomness
- Still maintains coherence and structure at 0.9

---

## Next: 05_evaluation_comparison.ipynb
Compare all three roasting styles side-by-side and evaluate their effectiveness.